#Assignment 2 – Knowledge Graph

Perform Named-Entity Recognition (NER) on the corpus documents

Identify, and list, the en es and their labels that are recognized by spaCy’s default
medium-sized model (‘en_core_web_md’);

In [12]:
from src.Project2.ner_baseline import run_baseline_ner, summarize, save_entity_frequencies_to_csv
df = run_baseline_ner("../../data/train")
save_entity_frequencies_to_csv(df, "shakespeare_entities.csv")
summarize(df)

Looking in: E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\data\train
Matched files:
 - Shakespeare_Macbeth.txt
 - Shakespeare_Midsummer_Nights_Dream.txt
 - Shakespeare_Much_Ado_About_Nothing.txt
 - Shakespeare_Romeo_and_Juliet.txt
Loaded 4 text files
Processing: Shakespeare_Macbeth.txt
Processing: Shakespeare_Midsummer_Nights_Dream.txt
Processing: Shakespeare_Much_Ado_About_Nothing.txt
Processing: Shakespeare_Romeo_and_Juliet.txt
Saved entity frequencies to: shakespeare_entities.csv

=== Label Distribution ===
label
PERSON         2091
ORG             753
GPE             366
CARDINAL        250
DATE            230
TIME            198
ORDINAL         119
NORP             64
WORK_OF_ART      48
PRODUCT          44
LOC              17
LANGUAGE          9
QUANTITY          8
MONEY             5
EVENT             5
FAC               4
LAW               1
Name: count, dtype: int64

=== Entities with Frequency ===
     

2. Identify and list, any en es that have been mislabeled by the default model;
3. Identify any en es that are missing labels, or do not have a default label to describe
their intended meaning;

In [13]:
from src.Project2.ner_mislabeled_finder import load_entity_csv, build_expected_entities, find_mislabeled_entities, find_missing_entities, find_no_good_default_label_entities

freq_df = load_entity_csv("shakespeare_entities.csv")
expected_entities = build_expected_entities()

mislabeled_df = find_mislabeled_entities(freq_df, expected_entities)
missing_df = find_missing_entities(freq_df, expected_entities)
no_default_df = find_no_good_default_label_entities(freq_df)

print("\n=== Step 2: Mislabeled Entities ===")
print(mislabeled_df.to_string(index=False))

print("\n=== Step 3A: Missing Entities ===")
print(missing_df.to_string(index=False))

print("\n=== Step 3B: No Good Default Label ===")
print(no_default_df.to_string(index=False))


=== Step 2: Mislabeled Entities ===
      entity predicted_label expected_label  count
     CLAUDIO             ORG         PERSON    104
       NURSE             GPE            OCC     87
       Romeo             ORG         PERSON     74
     CAPULET             ORG         PERSON     68
   DEMETRIUS             ORG         PERSON     43
       Paris             GPE         PERSON     34
        Hero             ORG         PERSON     30
      Helena             ORG         PERSON     24
   Demetrius             ORG         PERSON     23
       PARIS             GPE         PERSON     23
     TITANIA             ORG         PERSON     22
       Cupid             ORG         PERSON     20
    Beatrice             ORG         PERSON     17
      Hermia             GPE         PERSON     15
   MESSENGER          PERSON            OCC     15
    Montague             GPE         PERSON     15
        HERO             ORG         PERSON     14
    MONTAGUE            DATE         PERSON  

4. For all en es iden fied in step 3 above, fine-tune the model to correct any
- Mislabeled entities ,
- Missing-label entities ,
- Pronoun resolution, coreference resolution – go beyond the ‘1-sentence-back’ model to write your own custom code for pronoun resolution.

5. List out all the entities with their appropriate labels. Use a spreadsheet/table to include this informa on in your report

In [14]:
from src.Project2.ner_finetuning import (
    TRAIN_DATA,
    load_entity_frequency_csv,
    fine_tune_shakespeare_ner,
    resolve_pronouns_custom,
    export_final_entity_table,
    load_shakespeare_test_text,
    mine_training_data_from_corpus
)

freq_df = load_entity_frequency_csv("shakespeare_entities.csv")
freq_df.head()

,entity,label,count
0,BENEDICK,PERSON,133
1,PEDRO,PERSON,120
2,CLAUDIO,ORG,104
3,NURSE,GPE,87
4,Romeo,ORG,74


In [15]:
corpus_text = load_shakespeare_test_text("../../data/train")
expected_entities = build_expected_entities()

mined = mine_training_data_from_corpus(
    corpus_text=corpus_text,
    expected_entities=expected_entities,
    max_positive=250,
    max_negative=120,
    base_model="en_core_web_md",
    seed=42,
)

corpus_train_data = mined["all_examples"]

print("Custom TRAIN_DATA:", len(TRAIN_DATA))
print("Corpus-mined examples:", len(corpus_train_data))
print("Positive mined:", len(mined["positive_examples"]))
print("Negative mined:", len(mined["negative_examples"]))

Custom TRAIN_DATA: 89
Corpus-mined examples: 370
Positive mined: 250
Negative mined: 120


In [16]:
for i in range(10):
    text, ann = corpus_train_data[i]
    print(text)
    print(ann)
    print("-" * 80)

Snatching Romeo’s dagger.
{'entities': [(10, 15, 'PERSON')]}
--------------------------------------------------------------------------------
I married them; and their stol’n marriage day Was Tybalt’s doomsday, whose untimely death Banish’d the new-made bridegroom from this city;
{'entities': [(50, 56, 'PERSON')]}
--------------------------------------------------------------------------------
Stand forth, Lysander.
{'entities': [(13, 21, 'PERSON')]}
--------------------------------------------------------------------------------
Call Philostrate.
{'entities': [(5, 16, 'PERSON')]}
--------------------------------------------------------------------------------
Go you to Juliet ere you go to bed, Prepare her, wife, against this wedding day.
{'entities': [(10, 16, 'PERSON'), (49, 53, 'REL')]}
--------------------------------------------------------------------------------
Fleance his son, that keeps him company, Whose absence is no less material to me Than is his father’s, must embrace t

In [17]:
def merge_training_data(custom_train_data, corpus_train_data):
    merged = []
    seen = set()

    for text, ann in custom_train_data + corpus_train_data:
        key = (text, tuple(ann["entities"]))
        if key not in seen:
            seen.add(key)
            merged.append((text, ann))

    return merged

MERGED_TRAIN_DATA = merge_training_data(TRAIN_DATA, corpus_train_data)

print("Merged training examples:", len(MERGED_TRAIN_DATA))

Merged training examples: 459


In [18]:
nlp_ft = fine_tune_shakespeare_ner(
    train_data=MERGED_TRAIN_DATA,
    output_dir="shakespeare_ner_model_merged",
    base_model="en_core_web_md",
    n_iter=100,
    dropout=0.2,
    entity_labels=expected_entities,
    seed=42,
)

C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Speak, Pyramus.—Thisbe, stand forth." with entities "[(7, 14, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "She could have run and waddled all about; For even..." with entities "[(98, 105, 'REL')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities cou

Iteration 1/150 - Losses: {'ner': 752.7460003731663}
Iteration 2/150 - Losses: {'ner': 600.4121946487986}
Iteration 3/150 - Losses: {'ner': 531.3353762408609}
Iteration 4/150 - Losses: {'ner': 507.87002925604753}
Iteration 5/150 - Losses: {'ner': 470.58911263326206}
Iteration 6/150 - Losses: {'ner': 422.9249433447262}
Iteration 7/150 - Losses: {'ner': 380.77945174715444}
Iteration 8/150 - Losses: {'ner': 355.2375947722494}
Iteration 9/150 - Losses: {'ner': 339.99391371257616}
Iteration 10/150 - Losses: {'ner': 324.5436021746027}
Iteration 11/150 - Losses: {'ner': 303.50070994922135}
Iteration 12/150 - Losses: {'ner': 267.55358356066347}
Iteration 13/150 - Losses: {'ner': 271.47540569804096}
Iteration 14/150 - Losses: {'ner': 245.21222047458355}
Iteration 15/150 - Losses: {'ner': 223.29496779976944}
Iteration 16/150 - Losses: {'ner': 222.05407474134267}
Iteration 17/150 - Losses: {'ner': 225.29182526423506}
Iteration 18/150 - Losses: {'ner': 199.71516098497}
Iteration 19/150 - Losses: {

In [22]:
sample_text = """
Macbeth met Banquo in Scotland.
The Messenger warned Macbeth.
Romeo loved Juliet in Verona.
Friar Francis helped Hero.
First Witch greeted Macbeth.
"""

doc_sample = nlp_ft(sample_text)
[(ent.text, ent.label_) for ent in doc_sample.ents]

[('Macbeth', 'PERSON'),
 ('Banquo', 'PERSON'),
 ('Scotland', 'GPE'),
 ('Messenger', 'OCC'),
 ('Macbeth', 'PERSON'),
 ('Romeo', 'PERSON'),
 ('Juliet', 'PERSON'),
 ('Verona', 'GPE'),
 ('Friar Francis', 'PERSON'),
 ('Hero', 'PERSON'),
 ('First Witch', 'OCC'),
 ('Macbeth', 'PERSON')]

In [20]:
test_text = load_shakespeare_test_text("../../data/train")
doc_ft = nlp_ft(test_text)

final_df = export_final_entity_table(
    text=test_text,
    model_path="shakespeare_ner_model",
    output_csv="final_entities.csv"
)

Saved final entity table to E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\src\Project2\final_entities.csv


In [21]:
coref_df = resolve_pronouns_custom(doc_ft)

coref_df = coref_df.head(50)
coref_df

,pronoun,sentence,resolved_to,rationale
0,He,"He can report, As seemeth by his plight, of th...",DUNCAN,Most recent male-compatible entity in rolling ...
1,his,"He can report, As seemeth by his plight, of th...",DUNCAN,Most recent male-compatible entity in rolling ...
2,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
3,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
4,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
5,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
6,him,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
7,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
8,him,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
9,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
